In [1]:
# Importando as bibliotecas

import pandas as pd 
import numpy as np
import os
from datetime import datetime, date
from google.cloud import bigquery
from dotenv import load_dotenv
import pyarrow


# Carregando variáveis de ambiente
load_dotenv()

True

In [2]:
df_2025_01 = pd.read_csv("/home/danielpedro/Engenharia de Dados/combustivel_preco2015_2025/dataset/Preços semestrais - AUTOMOTIVOS_2025.01.csv", sep=';',low_memory=False) 

In [3]:
df_2025_02 = pd.read_csv("/home/danielpedro/Engenharia de Dados/combustivel_preco2015_2025/dataset/Preços semestrais - AUTOMOTIVOS_2025.02.csv", sep=';',low_memory=False)

In [4]:
df_2025 = pd.concat((df_2025_01, df_2025_02))

In [5]:
df_2025.info()

<class 'pandas.core.frame.DataFrame'>
Index: 813731 entries, 0 to 384207
Data columns (total 16 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   Regiao - Sigla     804617 non-null  object 
 1   Estado - Sigla     804617 non-null  object 
 2   Municipio          804617 non-null  object 
 3   Revenda            804617 non-null  object 
 4   CNPJ da Revenda    804617 non-null  object 
 5   Nome da Rua        804617 non-null  object 
 6   Numero Rua         804535 non-null  object 
 7   Complemento        183322 non-null  object 
 8   Bairro             803093 non-null  object 
 9   Cep                804617 non-null  object 
 10  Produto            804617 non-null  object 
 11  Data da Coleta     804617 non-null  object 
 12  Valor de Venda     804617 non-null  object 
 13  Valor de Compra    0 non-null       float64
 14  Unidade de Medida  804617 non-null  object 
 15  Bandeira           804617 non-null  object 
dtypes: floa

In [6]:
df_2025_pe = df_2025[df_2025["Estado - Sigla"] == "PE"]

In [7]:
df_2025_pe.head()

,Regiao - Sigla,Estado - Sigla,Municipio,Revenda,CNPJ da Revenda,Nome da Rua,Numero Rua,Complemento,Bairro,Cep,Produto,Data da Coleta,Valor de Venda,Valor de Compra,Unidade de Medida,Bandeira
816,NE,PE,PETROLINA,TROPICAL PETROLEO LTDA,08.941.887/0001-24,RODOVIA BR 428,S/N,NaN,ZONA RURAL,56300-990,GASOLINA,02/01/2025,"6,76",NaN,R$ / litro,PETROBAHIA
817,NE,PE,PETROLINA,TROPICAL PETROLEO LTDA,08.941.887/0001-24,RODOVIA BR 428,S/N,NaN,ZONA RURAL,56300-990,GASOLINA ADITIVADA,02/01/2025,"6,76",NaN,R$ / litro,PETROBAHIA
818,NE,PE,PETROLINA,TROPICAL PETROLEO LTDA,08.941.887/0001-24,RODOVIA BR 428,S/N,NaN,ZONA RURAL,56300-990,DIESEL,02/01/2025,"5,99",NaN,R$ / litro,PETROBAHIA
819,NE,PE,PETROLINA,TROPICAL PETROLEO LTDA,08.941.887/0001-24,RODOVIA BR 428,S/N,NaN,ZONA RURAL,56300-990,DIESEL S10,02/01/2025,"5,99",NaN,R$ / litro,PETROBAHIA
820,NE,PE,PETROLINA,TROPICAL PETROLEO LTDA,08.941.887/0001-24,RODOVIA BR 428,S/N,NaN,ZONA RURAL,56300-990,ETANOL,02/01/2025,"5,10",NaN,R$ / litro,PETROBAHIA


In [11]:
df_2025_pe['CNPJ da Revenda'] = df_2025_pe['CNPJ da Revenda'].astype(str).str.zfill(14)

/tmp/ipykernel_896/3411917950.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2025_pe['CNPJ da Revenda'] = df_2025_pe['CNPJ da Revenda'].astype(str).str.zfill(14)


In [ ]:
# Configurando credenciais

# Caminho do arquivo de credenciais GCP
credencial = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")

# Identificador do projeto no GCP
project_id = os.getenv("PROJECT_ID")

# Tabela Bronze no BigQuery
table_id = os.getenv("BRONZE_2025")

# Inicialização do cliente BigQuery

# Define a variavel de ambiente
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = credencial

# Cria o cliente Bigquery ao projeto configurado
client = bigquery.Client(project=project_id)

In [13]:
# Criação e carga da tabela BRONZE

# Configuração do job de carga no BigQuery
# WRITE_TRUNCATE garante que a tabela Bronze seja sempre sobrescrita,
# mantendo apenas o retrato mais recente dos dados de origem
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE"
)

# Envia o DataFrame para o BigQuery,
# criando a tabela automaticamente caso ainda não exista
job = client.load_table_from_dataframe(
    df_2025_pe,
    table_id,
    job_config=job_config
)

# Aguarda a finalização do job para garantir consistência da carga
job.result()

# Confirmação visual de sucesso da operação
print("✅ Tabela Bronze criada e dados carregados com sucesso")

/home/danielpedro/anaconda3/envs/combustivel_projeto/lib/python3.11/site-packages/google/cloud/bigquery/_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


✅ Tabela Bronze criada e dados carregados com sucesso
